<a href="https://colab.research.google.com/github/samuraiinst2025/github-basic-kadai/blob/main/Final/S%E6%A7%98001.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================
# ライブラリ読み込み
# ==============================

# 表形式データ（Excel・CSVなど）を扱うためのライブラリ
import pandas as pd

# 複数ファイルをまとめて取得するための標準ライブラリ
import glob

# ファイルパスを安全に組み立てるための標準ライブラリ
import os

# Google Colab で Google Drive を使うための専用ライブラリ
from google.colab import drive


# ==============================
# Google Drive をマウント
# ==============================

drive.mount('/content/drive')


# ==============================
# ① 各店の注文情報をまとめる
# ==============================

# 今日の日付（ファイル名に使われている形式）
today = "20230524"

# 注文ファイルがあるディレクトリ
ORDER_DIR = "/content/drive/MyDrive/samples/order_new"

# 今日の日付の注文ファイルだけ取得
pattern = f"order_*_{today}.xlsx"
order_files = glob.glob(os.path.join(ORDER_DIR, pattern))

print(f"対象ファイル数: {len(order_files)}")

# 合計注文数を入れるDataFrame
total_df = None

for file in order_files:
    df = pd.read_excel(file)

    if total_df is None:
        total_df = df.copy()
    else:
        total_df = total_df.add(df, fill_value=0)

# 注文ファイルが1件もなかった場合のエラー対策
if total_df is None:
    raise ValueError("本日の注文ファイルが見つかりませんでした")

print("▼ 各野菜の合計注文数")
display(total_df)


# ==============================
# ② 注文集計結果を保存
# ==============================

output_path = f"/content/drive/MyDrive/samples/summary_order_{today}.xlsx"
total_df.to_excel(output_path, index=False)

print(f"集計結果を保存しました: {output_path}")


# ==============================
# ③ 現在の在庫状況を確認
# ==============================

# 在庫表のパス
INVENTORY_PATH = "/content/drive/MyDrive/samples/inventory.xlsx"

# 在庫表を読み込む
inventory_df = pd.read_excel(INVENTORY_PATH)

# 最終行（最新在庫）
latest_inventory = inventory_df.iloc[-1]

print("▼ 最新の在庫情報（元データ）")
display(latest_inventory)

# 数値列（野菜）だけ抽出
latest_inventory_no_date = latest_inventory.drop(['日付', '曜日'])


# ==============================
# ④ 注文反映後の在庫を計算
# ==============================

# 各野菜の合計注文数（Series化）
total_order_series = total_df.sum()

# 残在庫 = 最新在庫 - 注文数
remaining_inventory = latest_inventory_no_date - total_order_series

print("▼ 注文反映後の在庫数")
display(remaining_inventory)


# ==============================
# ⑤ pickup.xlsx からしきい値・発注数を取得
# ==============================

PICKUP_PATH = "/content/drive/MyDrive/samples/pickup.xlsx"

# 1列目をindexとして読み込む
pickup_df = pd.read_excel(PICKUP_PATH, index_col=0)

print("▼ pickup.xlsx")
display(pickup_df)

# しきい値
threshold_series = pickup_df.loc['しきい値']

# 発注数（追加量）
order_qty_series = pickup_df.loc['追加量']


# ==============================
# ⑥ 発注が必要な野菜を特定
# ==============================

# 残在庫がしきい値を下回っているか
below_threshold = remaining_inventory < threshold_series.reindex(remaining_inventory.index)

print("▼ しきい値を下回っているか")
display(below_threshold)

# 発注対象の野菜
low_stock = remaining_inventory[below_threshold]

print("▼ 発注が必要な野菜（残在庫）")
display(low_stock)


# ==============================
# ⑦ 発注対象と発注数を紐づける
# ==============================

# 発注が必要な野菜名
order_items = low_stock.index

# 発注数を取得（ズレ防止で reindex）
order_list = order_qty_series.reindex(order_items)

print("▼ 発注対象と発注数")
display(order_list)


# ==============================
# ⑧ メール用 発注内容テキスト作成
# ==============================

order_lines = []

for veg, qty in order_list.items():
    line = f"・{veg}：{int(qty)} 個"
    order_lines.append(line)

order_text = "\n".join(order_lines)

print("▼ メール用 発注内容")
print(order_text)


# ==============================
# ⑨ 発注メール本文を生成
# ==============================

mail_body = f"""
〇〇農園 御中

いつもお世話になっております。
株式会社△△の□□です。

下記内容にて野菜の発注をお願いいたします。

【発注内容】
{order_text}

納品日：明日

ご不明点等ございましたらご連絡ください。
何卒よろしくお願いいたします。

――――――――――――
株式会社△△
□□
"""

print("▼ 発注メール本文")
print(mail_body)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
対象ファイル数: 4
▼ 各野菜の合計注文数


,ほうれん草,キャベツ,トマト,ニンジン,レタス,大根,白菜
0,23.0,21.0,31.0,32.0,42,15.0,25.0


集計結果を保存しました: /content/drive/MyDrive/samples/summary_order_20230524.xlsx
▼ 最新の在庫情報（元データ）


,1002
日付,2026-02-04 19:28:58
曜日,Wednesday
トマト,-91.0
キャベツ,17.0
レタス,-26.0
白菜,-15.0
ほうれん草,11.0
大根,15.0
ニンジン,-16.0


▼ 注文反映後の在庫数


...

▼ 発注が必要な野菜（残在庫）


,0
ほうれん草,-12.0
キャベツ,-4.0
トマト,-122.0
ニンジン,-48.0
レタス,-68.0
大根,0.0
白菜,-40.0


▼ 発注対象と発注数


,追加量
ほうれん草,80
キャベツ,80
トマト,2
ニンジン,80
レタス,100
大根,60
白菜,60


▼ メール用 発注内容
・ほうれん草：80 個
・キャベツ：80 個
・トマト：2 個
・ニンジン：80 個
・レタス：100 個
・大根：60 個
・白菜：60 個
▼ 発注メール本文

〇〇農園 御中

いつもお世話になっております。
株式会社△△の□□です。

下記内容にて野菜の発注をお願いいたします。

【発注内容】
・ほうれん草：80 個
・キャベツ：80 個
・トマト：2 個
・ニンジン：80 個
・レタス：100 個
・大根：60 個
・白菜：60 個

納品日：明日

ご不明点等ございましたらご連絡ください。
何卒よろしくお願いいたします。

――――――――――――
株式会社△△
□□



In [ ]:
from google.colab import drive
drive.mount('/content/drive')